# 01 - Data Understanding

Goal: load both raw datasets and understand their shape, types, and quality issues.
No cleaning happens in this notebook — inspection only.

## Load Online Retail II (both sheets)

In [1]:
import pandas as pd

df_2009_2010 = pd.read_excel(
    "../data/raw/online_retail_II.xlsx",
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    "../data/raw/online_retail_II.xlsx",
    sheet_name="Year 2010-2011"
)

df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)
df.shape

(1067371, 8)

Note: despite the sheet names ("2009-2010" and "2010-2011"), the combined
data actually spans **2009-12-01 to 2011-12-09** almost two full years.
Worth checking directly instead of trusting the sheet names.

In [2]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB


In [4]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

`Description` has 4,382 missing values, `Customer ID` has 243,007 missing.
Missing Customer ID is expected — not every transaction can be tied to a logged-in
customer. We keep these rows for transaction/product-level analysis and only drop
them later for customer-level work (RFM, segmentation).

In [5]:
df.duplicated().sum()

np.int64(34335)

34,335 exact duplicate rows. These are likely repeated line entries (same
invoice/product/qty/price/date logged twice) rather than meaningful repeat
purchases. They get dropped in the cleaning notebook.

In [6]:
df["InvoiceDate"].min(), df["InvoiceDate"].max()

(Timestamp('2009-12-01 07:45:00'), Timestamp('2011-12-09 12:50:00'))

In [7]:
df["Quantity"].describe()

count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64

Quantity has negative values (returns / cancellations) — expected, not an error.
Same logic applies to Price below.

In [8]:
df["Price"].describe()

count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64

In [9]:
(df["Quantity"] < 0).sum(), (df["Price"] < 0).sum(), (df["Price"] == 0).sum()

(np.int64(22950), np.int64(5), np.int64(6202))

In [10]:
df["Invoice"] = df["Invoice"].astype(str)
df["Invoice"].str.startswith("C").sum()

np.int64(19494)

19,494 invoices start with "C" — cancellations. These get removed in cleaning,
along with the negative-quantity and non-positive-price rows they mostly overlap with.

In [11]:
df["Country"].nunique(), df["Customer ID"].nunique(), df["StockCode"].nunique()

(43, 5942, 5305)

## Load Online Shoppers Purchasing Intention

In [12]:
shoppers = pd.read_csv("../data/raw/online_shoppers_intention.csv")
shoppers.shape

(12330, 18)

In [13]:
shoppers.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

In [14]:
shoppers.isnull().sum().sum()

np.int64(0)

In [15]:
shoppers["Revenue"].value_counts()

Revenue
False    10422
True      1908
Name: count, dtype: int64

Target is heavily imbalanced: 10,422 False vs 1,908 True (~15.5% positive rate).
This is the reason notebook 06 (purchase prediction) can't rely on accuracy alone.

In [16]:
shoppers.duplicated().sum()

np.int64(125)

125 duplicate rows. Unlike the retail dataset, there's no invoice/customer ID
here to distinguish a "real" repeat session from an accidental duplicate row —
these are behavioral feature vectors, not transaction records, so two identical
rows are almost certainly a logging duplicate, not two different real sessions.
Dropped in the next notebook.